# ML-Based Product Analysis Demo
## Demonstrating ML Capabilities for Price and Review Analysis

This notebook demonstrates what can be accomplished with Machine Learning using the tools available in this repository, simulating a product comparison system with sample data.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## 1. Creating Sample Refrigerator Data
Since I cannot access external shopping portals, I'll create realistic sample data to demonstrate ML capabilities.

In [ ]:
# Create sample refrigerator data
np.random.seed(42)

brands = ['Samsung', 'LG', 'Whirlpool', 'GE', 'Frigidaire', 'KitchenAid', 'Bosch']
types = ['Top Freezer', 'Bottom Freezer', 'Side by Side', 'French Door', 'Compact']
energy_ratings = ['A+++', 'A++', 'A+', 'A', 'B']

n_samples = 500

# Generate features
data = {
    'brand': np.random.choice(brands, n_samples),
    'type': np.random.choice(types, n_samples),
    'capacity_liters': np.random.normal(400, 150, n_samples).clip(150, 800),
    'energy_rating': np.random.choice(energy_ratings, n_samples),
    'width_cm': np.random.normal(60, 10, n_samples).clip(50, 90),
    'height_cm': np.random.normal(180, 20, n_samples).clip(150, 220),
    'has_ice_maker': np.random.choice([0, 1], n_samples, p=[0.6, 0.4]),
    'has_water_dispenser': np.random.choice([0, 1], n_samples, p=[0.7, 0.3]),
    'warranty_years': np.random.choice([1, 2, 3, 5], n_samples, p=[0.1, 0.3, 0.4, 0.2]),
    'avg_rating': np.random.normal(4.2, 0.8, n_samples).clip(1, 5),
    'num_reviews': np.random.exponential(50, n_samples).astype(int).clip(1, 500)
}

df = pd.DataFrame(data)

# Calculate price based on realistic factors
base_price = 500
brand_multiplier = {'Samsung': 1.2, 'LG': 1.15, 'Whirlpool': 1.0, 'GE': 1.1, 
                   'Frigidaire': 0.9, 'KitchenAid': 1.3, 'Bosch': 1.25}
type_multiplier = {'Top Freezer': 0.8, 'Bottom Freezer': 1.0, 'Side by Side': 1.1,
                  'French Door': 1.3, 'Compact': 0.6}
energy_multiplier = {'A+++': 1.2, 'A++': 1.1, 'A+': 1.05, 'A': 1.0, 'B': 0.95}

df['price'] = (
    base_price +
    df['capacity_liters'] * 1.5 +
    df['brand'].map(brand_multiplier) * 200 +
    df['type'].map(type_multiplier) * 300 +
    df['energy_rating'].map(energy_multiplier) * 100 +
    df['has_ice_maker'] * 150 +
    df['has_water_dispenser'] * 100 +
    df['warranty_years'] * 50 +
    np.random.normal(0, 100, n_samples)  # Add some noise
).round(0).astype(int)

print(f"Created dataset with {len(df)} refrigerators")
print("\nFirst few rows:")
df.head()

## 2. Price Prediction Model
Building a model to predict refrigerator prices based on features.

In [ ]:
# Prepare data for modeling
# Encode categorical variables
le_brand = LabelEncoder()
le_type = LabelEncoder()
le_energy = LabelEncoder()

df_model = df.copy()
df_model['brand_encoded'] = le_brand.fit_transform(df['brand'])
df_model['type_encoded'] = le_type.fit_transform(df['type'])
df_model['energy_encoded'] = le_energy.fit_transform(df['energy_rating'])

# Select features for prediction
features = ['capacity_liters', 'width_cm', 'height_cm', 'has_ice_maker',
           'has_water_dispenser', 'warranty_years', 'brand_encoded', 
           'type_encoded', 'energy_encoded']

X = df_model[features]
y = df_model['price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    # Train model
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'mse': mse,
        'rmse': np.sqrt(mse),
        'r2': r2,
        'predictions': y_pred
    }
    
    print(f"\n{name} Results:")
    print(f"RMSE: ${mse**0.5:.2f}")
    print(f"R² Score: {r2:.3f}")

## 3. Best Value Analysis
Finding refrigerators with the best value (high rating, low price).

In [ ]:
# Calculate value score (higher rating, lower price = better value)
# Normalize price and rating for fair comparison
price_normalized = (df['price'] - df['price'].min()) / (df['price'].max() - df['price'].min())
rating_normalized = (df['avg_rating'] - df['avg_rating'].min()) / (df['avg_rating'].max() - df['avg_rating'].min())

# Value score: high rating, low price
df['value_score'] = rating_normalized - price_normalized

# Find best value refrigerators
best_value = df.nlargest(10, 'value_score')[['brand', 'type', 'capacity_liters', 
                                            'price', 'avg_rating', 'value_score']]

print("Top 10 Best Value Refrigerators:")
print(best_value)

## 4. Price vs Features Visualization
Understanding how different features affect pricing.

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Price by brand
df.boxplot(column='price', by='brand', ax=axes[0,0])
axes[0,0].set_title('Price Distribution by Brand')
axes[0,0].set_xlabel('Brand')
axes[0,0].set_ylabel('Price ($)')
plt.setp(axes[0,0].xaxis.get_majorticklabels(), rotation=45)

# Price vs Capacity
axes[0,1].scatter(df['capacity_liters'], df['price'], alpha=0.6)
axes[0,1].set_title('Price vs Capacity')
axes[0,1].set_xlabel('Capacity (Liters)')
axes[0,1].set_ylabel('Price ($)')

# Rating vs Price
axes[1,0].scatter(df['price'], df['avg_rating'], alpha=0.6)
axes[1,0].set_title('Rating vs Price')
axes[1,0].set_xlabel('Price ($)')
axes[1,0].set_ylabel('Average Rating')

# Value score distribution
axes[1,1].hist(df['value_score'], bins=30, alpha=0.7)
axes[1,1].set_title('Value Score Distribution')
axes[1,1].set_xlabel('Value Score')
axes[1,1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 5. Feature Importance Analysis
Understanding which features most influence price.

In [ ]:
# Feature importance from Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.title('Feature Importance for Price Prediction')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Feature Importance Ranking:")
print(feature_importance)

## 6. Recommendation Function
A function to recommend refrigerators based on user preferences.

In [ ]:
def recommend_refrigerator(budget_max=None, min_capacity=None, preferred_brand=None, 
                          min_rating=None, must_have_ice_maker=False, 
                          must_have_water_dispenser=False, top_n=5):
    """
    Recommend refrigerators based on user criteria.
    
    This demonstrates what an ML-powered recommendation system could look like
    if we had access to real product data.
    """
    filtered_df = df.copy()
    
    # Apply filters
    if budget_max:
        filtered_df = filtered_df[filtered_df['price'] <= budget_max]
    
    if min_capacity:
        filtered_df = filtered_df[filtered_df['capacity_liters'] >= min_capacity]
    
    if preferred_brand:
        filtered_df = filtered_df[filtered_df['brand'] == preferred_brand]
    
    if min_rating:
        filtered_df = filtered_df[filtered_df['avg_rating'] >= min_rating]
    
    if must_have_ice_maker:
        filtered_df = filtered_df[filtered_df['has_ice_maker'] == 1]
    
    if must_have_water_dispenser:
        filtered_df = filtered_df[filtered_df['has_water_dispenser'] == 1]
    
    if len(filtered_df) == 0:
        return "No refrigerators match your criteria. Please adjust your filters."
    
    # Sort by value score
    recommendations = filtered_df.nlargest(top_n, 'value_score')
    
    return recommendations[['brand', 'type', 'capacity_liters', 'price', 
                           'avg_rating', 'has_ice_maker', 'has_water_dispenser',
                           'value_score']]

# Example usage
print("Example 1: Budget under $1000, minimum 300L capacity, rating > 4.0")
rec1 = recommend_refrigerator(budget_max=1000, min_capacity=300, min_rating=4.0)
print(rec1)

print("\n" + "="*80 + "\n")

print("Example 2: Samsung brand, must have ice maker and water dispenser")
rec2 = recommend_refrigerator(preferred_brand='Samsung', 
                             must_have_ice_maker=True, 
                             must_have_water_dispenser=True)
print(rec2)

## Summary

This demonstration shows what **CAN** be accomplished with Machine Learning using the tools available in this repository:

### ✅ What We Demonstrated:
1. **Price Prediction Models** - Using features to predict refrigerator prices
2. **Value Analysis** - Finding best value products based on price and ratings
3. **Feature Importance** - Understanding which factors most affect pricing
4. **Recommendation System** - Filtering and ranking products by user preferences
5. **Data Visualization** - Creating insightful charts and graphs

### ❌ What We CANNOT Do:
- Access real shopping websites or APIs
- Scrape live product data
- Get real-time pricing information
- Access actual customer reviews

### 🎯 Real-World Application:
If you had access to product data (through APIs, datasets, or manual collection), you could use these exact techniques to build a comprehensive product comparison and recommendation system!

The ML models and analysis shown here represent the **core capabilities** that would power such a system.